In [ ]:
!git clone https://github.com/fashn-AI/fashn-vton-1.5.git
%cd fashn-vton-1.5

!pip install -e .
!pip install gradio

!python scripts/download_weights.py --weights-dir ./weights

In [ ]:
import os
import sys

if not os.getcwd().endswith("fashn-vton-1.5"):
    os.chdir("/content/fashn-vton-1.5")

sys.path.append(os.path.abspath("./src"))


import gradio as gr
from PIL import Image
from fashn_vton import TryOnPipeline

print("جاري تحميل الموديل... يرجى الانتظار (قد يستغرق بعض الوقت)")
pipeline = TryOnPipeline(weights_dir="./weights")
print("تم تحميل الموديل بنجاح! ")

def process_tryon(person_image, garment_image, category):
    if person_image is None or garment_image is None:
        return None

    person = person_image.convert("RGB")
    garment = garment_image.convert("RGB")

    result = pipeline(
        person_image=person,
        garment_image=garment,
        category=category,
    )
    return result.images[0]

with gr.Blocks(title="FASHN VTON 1.5 - Virtual Try-On") as demo:
    gr.Markdown("# 👕 تجربة الملابس افتراضياً - FASHN VTON 1.5")
    gr.Markdown("قم برفع صورة لشخص وصورة لقطعة ملابس، ثم اختر نوع الملابس واضغط على زر المعالجة.")

    with gr.Row():
        with gr.Column():
            person_img = gr.Image(type="pil", label="صورة الشخص")
            garment_img = gr.Image(type="pil", label="صورة قطعة الملابس")
            category_dropdown = gr.Dropdown(
                choices=["tops", "bottoms", "one-pieces"],
                value="tops",
                label="نوع الملابس"
            )
            submit_btn = gr.Button("✨ دمج الصور (جرب الآن)", variant="primary")

        with gr.Column():
            output_img = gr.Image(label="النتيجة النهائية")

    submit_btn.click(
        fn=process_tryon,
        inputs=[person_img, garment_img, category_dropdown],
        outputs=output_img
    )

demo.launch(share=True, debug=True)